## **Tareas a resolver:**

**Clasificación binaria: mentira o verdad**

**Predicción del hablante: de qué país es el mensaje**

### **Transformers**


##### **Tarea 1: predicción veracidad**


In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -------------------------------------
# Configuración Global
# -------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 10         # Aumentado a 10 como pediste
PATIENCE = 3        # Early Stopping: Si no mejora en 3 épocas, paramos
LR = 2e-5
MAX_LEN = 128
DATA_PATH = "data/train_preprocessed.parquet"

models_to_compare = [
    "distilbert-base-uncased",
    "distilroberta-base"
]

print(f"Usando dispositivo: {DEVICE}")

# -------------------------------------
# 1. Carga y Balanceo de Datos (Oversampling)
# -------------------------------------
df = pd.read_parquet(DATA_PATH)
df = df[df["sender_labels"].astype(str).str.lower().isin(["true", "false"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0}).values

le = LabelEncoder()
df['target'] = le.fit_transform(y_raw)

# Split
X_train_raw, X_val_raw, y_train_raw, y_val = train_test_split(
    df["messages"], df["target"], stratify=df["target"], test_size=0.2, random_state=42
)

# Oversampling en Train
train_df = pd.DataFrame({'text': X_train_raw, 'label': y_train_raw})
df_false = train_df[train_df['label'] == 0]
df_true = train_df[train_df['label'] == 1]

# Igualamos False a True
df_false_over = df_false.sample(len(df_true), replace=True, random_state=42)
df_balanced = pd.concat([df_true, df_false_over], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

X_train = df_balanced['text'].values
y_train = df_balanced['label'].values
X_val = X_val_raw.values
y_val = y_val.values

print(f"Datos preparados. Train (Balanceado): {len(X_train)} | Val: {len(X_val)}")

# -------------------------------------
# 2. Clases Utilitarias
# -------------------------------------
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_model(model_name, X_train, y_train, X_val, y_val):
    print(f"\n{'='*40}")
    print(f"PROCESANDO: {model_name}")
    print(f"{'='*40}")
    
    # Cargar Tokenizer y Modelo específicos
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to(DEVICE)
    
    # Dataloaders
    train_ds = TransformerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = TransformerDataset(X_val, y_val, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    optimizer = AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    
    # Variables de control
    best_mcc = -1
    best_epoch = 0
    patience_counter = 0
    history = []
    
    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            model.zero_grad()
            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- Validation ---
        model.eval()
        y_true, y_pred = [], []
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask=mask)
                loss = loss_fn(outputs.logits, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                y_pred.extend(preds)
                y_true.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        mcc = matthews_corrcoef(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="macro")
        acc = accuracy_score(y_true, y_pred)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Val Loss: {avg_val_loss:.4f} | MCC: {mcc:.4f} | F1: {f1:.4f}")
        
        history.append({
            "Model": model_name,
            "Epoch": epoch + 1,
            "Val Loss": avg_val_loss,
            "MCC": mcc,
            "F1": f1,
            "Accuracy": acc
        })
        
        # --- Early Stopping & Checkpoint ---
        if mcc > best_mcc:
            best_mcc = mcc
            best_epoch = epoch + 1
            patience_counter = 0
            # Aquí podrías guardar el modelo: torch.save(model.state_dict(), f"{model_name}_best.pt")
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print(f"Early Stopping activado. No mejora desde Epoch {best_epoch}.")
            break
            
    print(f"Mejor MCC para {model_name}: {best_mcc:.4f} (Epoch {best_epoch})")
    return history

# -------------------------------------
# 3. Ejecución y Comparativa
# -------------------------------------
all_results = []

for m in models_to_compare:
    res = train_model(m, X_train, y_train, X_val, y_val)
    all_results.extend(res)

# Crear DataFrame final
df_res = pd.DataFrame(all_results)
print("\n" + "="*50)
print("TABLA COMPARATIVA FINAL")
print("="*50)

# Mostrar la mejor fila de cada modelo (basado en MCC)
best_rows = df_res.loc[df_res.groupby("Model")["MCC"].idxmax()].sort_values("MCC", ascending=False)
print(best_rows[["Model", "Epoch", "Accuracy", "F1", "MCC"]])

Usando dispositivo: cuda
Datos preparados. Train (Balanceado): 18194 | Val: 2379

PROCESANDO: distilbert-base-uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10 | Val Loss: 0.4026 | MCC: 0.0573 | F1: 0.5276
Epoch 2/10 | Val Loss: 0.4854 | MCC: 0.0628 | F1: 0.5308
Epoch 3/10 | Val Loss: 0.5940 | MCC: 0.0668 | F1: 0.5329
Epoch 4/10 | Val Loss: 0.5596 | MCC: 0.0253 | F1: 0.5124
Epoch 5/10 | Val Loss: 0.5690 | MCC: 0.0790 | F1: 0.5395
Epoch 6/10 | Val Loss: 0.4614 | MCC: 0.0382 | F1: 0.5178
Epoch 7/10 | Val Loss: 0.5046 | MCC: 0.0424 | F1: 0.5194
Epoch 8/10 | Val Loss: 0.4310 | MCC: 0.0609 | F1: 0.5224
Early Stopping activado. No mejora desde Epoch 5.
Mejor MCC para distilbert-base-uncased: 0.0790 (Epoch 5)

PROCESANDO: distilroberta-base


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10 | Val Loss: 0.6459 | MCC: 0.0664 | F1: 0.5031
Epoch 2/10 | Val Loss: 0.5894 | MCC: 0.0808 | F1: 0.5396
Epoch 3/10 | Val Loss: 0.7481 | MCC: 0.1190 | F1: 0.5514
Epoch 4/10 | Val Loss: 0.6199 | MCC: 0.0750 | F1: 0.5373
Epoch 5/10 | Val Loss: 0.5200 | MCC: 0.0719 | F1: 0.5359
Epoch 6/10 | Val Loss: 0.5618 | MCC: 0.0670 | F1: 0.5332
Early Stopping activado. No mejora desde Epoch 3.
Mejor MCC para distilroberta-base: 0.1190 (Epoch 3)

TABLA COMPARATIVA FINAL
                      Model  Epoch  Accuracy        F1       MCC
10       distilroberta-base      3  0.887768  0.551430  0.119035
4   distilbert-base-uncased      5  0.924758  0.539465  0.078980


DistilRoBERTa presentó el desempeño más robusto (MCC 0.1190 y F1 0.5514). Aunque DistilBERT alcanzó una exactitud del 92.47%, su bajo MCC (0.0790) indica un claro sesgo hacia la clase mayoritaria (True), fallando en la identificación de mentiras. Por esto, el coeficiente de Matthews es la métrica óptima para esta tarea. El oversampling resulta insuficiente para modelar la complejidad del engaño sin técnicas complementarias.

Para intentar reducir el sesgo, reemplazamos el oversampling por Class Weights, penalizando severamente los errores en la clase minoritaria dentro de la loss function. También ajustamos el umbral de decisión para decidir si es una mentira (más de 0.5) para maximizar directamente el MCC.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight # <--- CAMBIO: Necesario para calcular pesos
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# -------------------------------------
# Configuración Global
# -------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 10
PATIENCE = 3
LR = 2e-5
MAX_LEN = 128
DATA_PATH = "data/train_preprocessed.parquet"

models_to_compare = ["distilroberta-base"] # Probamos con el mejor de la ronda anterior

print(f"Usando dispositivo: {DEVICE}")

# -------------------------------------
# 1. Carga de Datos (SIN OVERSAMPLING)
# -------------------------------------
df = pd.read_parquet(DATA_PATH)
df = df[df["sender_labels"].astype(str).str.lower().isin(["true", "false"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0}).values

le = LabelEncoder()
df['target'] = le.fit_transform(y_raw)

# Split normal (Stratified)
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    df["messages"], df["target"], stratify=df["target"], test_size=0.2, random_state=42
)

# <--- CAMBIO IMPORTANTE: Usamos los datos RAW, no los balanceados
X_train = X_train_raw.values
y_train = y_train_raw.values
X_val = X_val_raw.values
y_val = y_val_raw.values

print(f"Datos preparados (Originales). Train: {len(X_train)} | Val: {len(X_val)}")

# -------------------------------------
# 2. Calcular Pesos de Clase (Propuesta 1)
# -------------------------------------
# Calculamos el peso inverso para compensar el desbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
# Convertimos a tensor float y movemos al dispositivo
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print(f"Pesos calculados: Clase 0 (False): {class_weights[0]:.2f} | Clase 1 (True): {class_weights[1]:.2f}")
# Probablemente verás algo como: Clase 0: ~10.0 | Clase 1: ~0.5

# -------------------------------------
# 3. Clases Utilitarias (Dataset igual que antes)
# -------------------------------------
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# -------------------------------------
# 4. Función Auxiliar para Threshold Moving (Propuesta 2)
# -------------------------------------
def find_best_threshold(y_true, probs_class_0):
    """
    Busca el umbral óptimo para clasificar la clase minoritaria (0).
    Si prob_clase_0 > threshold -> Predice 0 (False)
    """
    best_thresh = 0.5
    best_mcc = -1
    
    thresholds = np.arange(0.1, 0.9, 0.05) # Probamos de 0.1 a 0.9
    
    for thresh in thresholds:
        # Si la probabilidad de ser 0 es mayor al umbral, es 0. Si no, es 1.
        preds = np.where(probs_class_0 > thresh, 0, 1)
        mcc = matthews_corrcoef(y_true, preds)
        
        if mcc > best_mcc:
            best_mcc = mcc
            best_thresh = thresh
            
    return best_thresh, best_mcc

# -------------------------------------
# 5. Loop de Entrenamiento Modificado
# -------------------------------------
def train_model_weighted(model_name, X_train, y_train, X_val, y_val, class_weights_tensor):
    print(f"\n{'='*40}")
    print(f"ENTRENANDO CON CLASS WEIGHTS: {model_name}")
    print(f"{'='*40}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to(DEVICE)
    
    train_ds = TransformerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = TransformerDataset(X_val, y_val, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    optimizer = AdamW(model.parameters(), lr=LR)
    
    # <--- CAMBIO: Aplicamos los pesos a la función de pérdida
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
    
    best_mcc_val = -1
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            model.zero_grad()
            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- Validation (con Probabilidades) ---
        model.eval()
        y_true_list = []
        probs_class_0_list = [] # Guardamos probabilidad de ser "False" (Clase 0)
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask=mask)
                loss = loss_fn(outputs.logits, labels)
                val_loss += loss.item()
                
                # <--- CAMBIO: Obtenemos probabilidades con Softmax
                probs = F.softmax(outputs.logits, dim=1)
                
                # Guardamos probabilidad de la clase 0 (la columna 0)
                probs_class_0_list.extend(probs[:, 0].cpu().numpy())
                y_true_list.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        
        # Conversión a arrays numpy
        y_true_np = np.array(y_true_list)
        probs_0_np = np.array(probs_class_0_list)
        
        # <--- CAMBIO: Buscamos el mejor umbral dinámicamente
        best_thresh, current_mcc = find_best_threshold(y_true_np, probs_0_np)
        
        # Generamos predicciones finales con ese umbral para las otras métricas
        final_preds = np.where(probs_0_np > best_thresh, 0, 1)
        f1 = f1_score(y_true_np, final_preds, average="macro")
        acc = accuracy_score(y_true_np, final_preds)
        
        print(f"Epoch {epoch+1} | Loss: {avg_val_loss:.4f} | Best MCC: {current_mcc:.4f} (Thresh: {best_thresh:.2f}) | F1: {f1:.4f}")
        
        # Early Stopping basado en MCC
        if current_mcc > best_mcc_val:
            best_mcc_val = current_mcc
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print("Early Stopping activado.")
            break

    print(f"Mejor MCC alcanzado: {best_mcc_val:.4f}")

# -------------------------------------
# 6. Ejecución
# -------------------------------------
for m in models_to_compare:
    train_model_weighted(m, X_train, y_train, X_val, y_val, weights_tensor)

Usando dispositivo: cuda
Datos preparados (Originales). Train: 9515 | Val: 2379
Pesos calculados: Clase 0 (False): 11.38 | Clase 1 (True): 0.52

ENTRENANDO CON CLASS WEIGHTS: distilroberta-base


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 1.1443 | Best MCC: 0.0000 (Thresh: 0.10) | F1: 0.4888
Epoch 2 | Loss: 0.8907 | Best MCC: 0.0837 (Thresh: 0.10) | F1: 0.5303
Epoch 3 | Loss: 0.7070 | Best MCC: 0.1083 (Thresh: 0.40) | F1: 0.5076
Epoch 4 | Loss: 0.9068 | Best MCC: 0.1232 (Thresh: 0.25) | F1: 0.5537
Epoch 5 | Loss: 1.1176 | Best MCC: 0.1130 (Thresh: 0.10) | F1: 0.5546
Epoch 6 | Loss: 1.2189 | Best MCC: 0.1113 (Thresh: 0.60) | F1: 0.5521
Epoch 7 | Loss: 1.7343 | Best MCC: 0.1038 (Thresh: 0.10) | F1: 0.5506
Early Stopping activado.
Mejor MCC alcanzado: 0.1232


Con esta prueba logramos subir el MCC a 0.1232, que es nuestro mejor resultado hasta ahora, pero la mejora ha sido muy pequeña. A partir del tercer epoch el loss se dispara, lo que significa que está empezando a memorizar en lugar de aprender.

El modelo intenta predecir si una frase es verdad o mentira, pero sin mas cotexto la misma frase puede ser verdad o mentira. Por eso a continuación vamos a añadir los sender y receiver labels. En el preprocesado añadimos una columna con el contexto que contiene datos con este formato: 

Emisor -> Receptor: Mensaje

In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# -------------------------------------
# Configuración Global
# -------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 10
PATIENCE = 3
LR = 2e-5
MAX_LEN = 128
# <--- CAMBIO 1: Ruta al nuevo archivo con contexto
DATA_PATH = "data/train_with_context.parquet" 

models_to_compare = ["distilroberta-base"]

print(f"Usando dispositivo: {DEVICE}")

# -------------------------------------
# 1. Carga de Datos (Con Contexto)
# -------------------------------------
df = pd.read_parquet(DATA_PATH)
df = df[df["sender_labels"].astype(str).str.lower().isin(["true", "false"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0}).values

le = LabelEncoder()
df['target'] = le.fit_transform(y_raw)

# <--- CAMBIO 2: Usamos la columna 'text_context' en lugar de 'messages'
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    df["text_context"], df["target"], stratify=df["target"], test_size=0.2, random_state=42
)

X_train = X_train_raw.values
y_train = y_train_raw.values
X_val = X_val_raw.values
y_val = y_val_raw.values

print(f"Datos preparados (Contexto Inyectado). Train: {len(X_train)} | Val: {len(X_val)}")
# Muestra un ejemplo para verificar
print(f"Ejemplo de entrada: {X_train[0]}")

# -------------------------------------
# 2. Calcular Pesos de Clase
# -------------------------------------
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f"Pesos: Clase 0 (False): {class_weights[0]:.2f} | Clase 1 (True): {class_weights[1]:.2f}")

# -------------------------------------
# 3. Dataset y Funciones
# -------------------------------------
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def find_best_threshold(y_true, probs_class_0):
    best_thresh = 0.5
    best_mcc = -1
    thresholds = np.arange(0.1, 0.9, 0.05)
    
    for thresh in thresholds:
        preds = np.where(probs_class_0 > thresh, 0, 1)
        mcc = matthews_corrcoef(y_true, preds)
        if mcc > best_mcc:
            best_mcc = mcc
            best_thresh = thresh
    return best_thresh, best_mcc

# -------------------------------------
# 4. Loop de Entrenamiento
# -------------------------------------
def train_model_weighted(model_name, X_train, y_train, X_val, y_val, class_weights_tensor):
    print(f"\n{'='*40}")
    print(f"ENTRENANDO (Context + Weights): {model_name}")
    print(f"{'='*40}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to(DEVICE)
    
    train_ds = TransformerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = TransformerDataset(X_val, y_val, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    optimizer = AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
    
    best_mcc_val = -1
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            model.zero_grad()
            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- Validation ---
        model.eval()
        y_true_list = []
        probs_class_0_list = []
        val_loss = 0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask=mask)
                loss = loss_fn(outputs.logits, labels)
                val_loss += loss.item()
                
                probs = F.softmax(outputs.logits, dim=1)
                probs_class_0_list.extend(probs[:, 0].cpu().numpy())
                y_true_list.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        y_true_np = np.array(y_true_list)
        probs_0_np = np.array(probs_class_0_list)
        
        best_thresh, current_mcc = find_best_threshold(y_true_np, probs_0_np)
        
        # Métricas con el mejor threshold
        final_preds = np.where(probs_0_np > best_thresh, 0, 1)
        f1 = f1_score(y_true_np, final_preds, average="macro")
        
        print(f"Epoch {epoch+1} | Loss: {avg_val_loss:.4f} | Best MCC: {current_mcc:.4f} (Thresh: {best_thresh:.2f}) | F1: {f1:.4f}")
        
        if current_mcc > best_mcc_val:
            best_mcc_val = current_mcc
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print("Early Stopping activado.")
            break

    print(f"Mejor MCC Final: {best_mcc_val:.4f}")

# -------------------------------------
# 5. Ejecución
# -------------------------------------
for m in models_to_compare:
    train_model_weighted(m, X_train, y_train, X_val, y_val, weights_tensor)


ENTRENANDO (Stabilized): distilroberta-base | LR: 5e-06


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 | Loss: 1.2037 | Best MCC: 0.0000 (Thresh: 0.15) | F1: 0.4888
Epoch 2 | Loss: 1.0305 | Best MCC: 0.1844 (Thresh: 0.15) | F1: 0.5908
Epoch 3 | Loss: 0.8772 | Best MCC: 0.1892 (Thresh: 0.25) | F1: 0.5913
Epoch 4 | Loss: 0.9336 | Best MCC: 0.1759 (Thresh: 0.10) | F1: 0.5850
Epoch 5 | Loss: 1.1102 | Best MCC: 0.1550 (Thresh: 0.10) | F1: 0.5705
Epoch 6 | Loss: 1.2296 | Best MCC: 0.1866 (Thresh: 0.15) | F1: 0.5932
Epoch 7 | Loss: 1.3164 | Best MCC: 0.1617 (Thresh: 0.45) | F1: 0.5725
Early Stopping activado.
Mejor MCC Final: 0.1892


##### **Tarea 2: predicción del hablante**


In [2]:
pip install transformers

  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.0 MB 2.8 MB/s eta 0:00:05
   ---- ----------------------------------- 1.3/12.0 MB 2.8 MB/s eta 0:00:04
   ------ --------------------------------- 1.8/12.0 MB 2.6 MB/s eta 0:00:04
   ------- -------------------------------- 2.4/12.0 MB 2.5 MB/s eta 0:00:04
   --------- ------------------------------ 2.9/12.0 MB 2.5 MB/s eta 0:00:04
   ----------- ---------------------------- 3.4/12.0 MB 2.5 MB/s eta 0:00:04
   ----------- ---------------------------- 3.4/12.0 MB 2.5 MB/s eta 0:00:04
   ----------- ---------------------------- 3.4/12.0 MB 2.5 MB/s eta 0:00:04
   ----------- ---------------------------- 3.4/12.0 MB 2.5 MB/s eta 0:00:04
   ------------ --------------------------- 3.7/12.0 MB 1.6 MB/s eta 0:00:06
   ------------

In [5]:
pip install hf_xet

  Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl.metadata (5.0 kB)
Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl (2.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =============================================
# CONFIG (optimizado para CPU)
# =============================================
DEVICE = "cpu"
torch.set_num_threads(4)

BATCH_SIZE = 8       # CPU-friendly
EPOCHS = 6           # suficiente, transformers convergen rápido
PATIENCE = 2
LR = 2e-5
MAX_LEN = 128

DATA_PATH = "data/train_preprocessed.parquet"

models_to_compare = [
    "distilbert-base-uncased",
    "distilroberta-base"
]

print(f"Usando dispositivo: {DEVICE}")

# =============================================
# 1. Cargar dataset
# =============================================
df = pd.read_parquet(DATA_PATH)

def flatten_speaker(x):
    if isinstance(x, list):
        return str(x[0])
    if isinstance(x, str) and x.startswith("["):
        return x.strip("[]").replace("'", "").split(",")[0].strip()
    return str(x)

df["speakers"] = df["speakers"].apply(flatten_speaker)

# Codificar etiquetas
le = LabelEncoder()
y = le.fit_transform(df["speakers"])
num_classes = len(le.classes_)

X = df["messages"].astype(str).values

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# Class weights
class_weights = compute_class_weight("balanced", classes=np.unique(y), y=y)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

# =============================================
# 2. Dataset
# =============================================
class SpeakerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        t = str(self.texts[idx])
        label = self.labels[idx]
        enc = self.tokenizer.encode_plus(
            t, add_special_tokens=True, max_length=self.max_len,
            padding="max_length", truncation=True,
            return_attention_mask=True, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# =============================================
# 3. Entrenamiento
# =============================================
def train_model(model_name):
    print("\n" + "="*50)
    print(f" ENTRENANDO {model_name} ")
    print("="*50)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes
    )
    model.to(DEVICE)

    train_ds = SpeakerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = SpeakerDataset(X_val, y_val, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    optimizer = AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    best_f1 = -1
    patience_counter = 0
    results = []

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # -------------------
        # VALIDACIÓN
        # -------------------
        model.eval()
        y_true, y_pred = [], []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                logits = model(input_ids, attention_mask=mask).logits
                preds = torch.argmax(logits, dim=1).cpu().numpy()

                y_pred.extend(preds)
                y_true.extend(labels.cpu().numpy())

        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="macro")

        print(f"Epoch {epoch+1}/{EPOCHS} | Acc={acc:.4f} | F1_macro={f1:.4f}")

        results.append({
            "Model": model_name,
            "Epoch": epoch+1,
            "Accuracy": acc,
            "F1_macro": f1
        })

        # -------------------
        # EARLY STOPPING
        # -------------------
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print("Early stopping activado.")
            break

    print(f"Mejor F1 para {model_name}: {best_f1:.4f}")
    return results

# =============================================
# 4. Ejecutar experimentos
# =============================================
all_results = []

for model_name in models_to_compare:
    r = train_model(model_name)
    all_results.extend(r)

# Tabla final
df_res = pd.DataFrame(all_results)
best = df_res.loc[df_res.groupby("Model")["F1_macro"].idxmax()]
print("\nRESULTADOS FINALES\n")
print(best)

Usando dispositivo: cpu

 ENTRENANDO distilbert-base-uncased 


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

C:\Users\Oihane\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Oihane\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of DistilB

Epoch 1/6 | Acc=0.2984 | F1_macro=0.2647
Epoch 2/6 | Acc=0.3615 | F1_macro=0.3172
Epoch 3/6 | Acc=0.3598 | F1_macro=0.3172
Epoch 4/6 | Acc=0.3615 | F1_macro=0.3235
Epoch 5/6 | Acc=0.3602 | F1_macro=0.3221
Epoch 6/6 | Acc=0.3472 | F1_macro=0.3053
Early stopping activado.
Mejor F1 para distilbert-base-uncased: 0.3235

 ENTRENANDO distilroberta-base 


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/6 | Acc=0.2762 | F1_macro=0.2580
Epoch 2/6 | Acc=0.3127 | F1_macro=0.2809
Epoch 3/6 | Acc=0.3577 | F1_macro=0.3184
Epoch 4/6 | Acc=0.3695 | F1_macro=0.3288
Epoch 5/6 | Acc=0.3380 | F1_macro=0.3179
Epoch 6/6 | Acc=0.3653 | F1_macro=0.3159
Early stopping activado.
Mejor F1 para distilroberta-base: 0.3288

RESULTADOS FINALES

                     Model  Epoch  Accuracy  F1_macro
3  distilbert-base-uncased      4  0.361496  0.323509
9       distilroberta-base      4  0.369483  0.328763


### **Otros modelos**


##### **Tarea 1: predicción veracidad**


##### **Tarea 2: predicción del hablante**
